# 48. LLM Parallelization + Orchestration Foundation

## Purpose
Parallelize the LLM calls inside `iterative_fix_loop` (via `batch_iterative_fix_loop`) to
cut runtime, and use that as the foundation for adding orchestration (e.g. a verifier agent)
on top.

## Background
- Preliminary proposal submitted. Now in the finals prep period.
- Notebook 45: `Aliphatic_long_chain` rule fully resolved (methyl-branching strategy).
- Notebooks 46-47: `PRECEDENT_LIBRARY` expanded from 11 to 23 entries (all 10 priority rules
  from valid-set frequency ranking covered).
- Qwen (`qwen3.8-max`) 300-sample rule-based vs LLM comparison: only 2 result differences,
  both correct (LLM applied precedents properly); 5 hold-for-review cases all matched
  precedent logic — confirms precedents genuinely improve LLM judgment quality.
- However execution is very slow (300 molecules took 3 hours, 43% timeout rate) — suspected
  cause: `qwen3.8-max` is a reasoning model whose internal thinking tokens aren't capped by
  `max_tokens`.
- Added `timeout=30` + error logging to `_call_llm`'s openai_compatible branch, and wrote
  `batch_iterative_fix_loop` (ThreadPoolExecutor-based) in `molecule_editor.py` — but the
  previous notebook got into a broken state (reload/commit confusion) so it's unclear
  whether these changes actually landed. **Needs to be re-verified from scratch in this
  notebook.**

## This session's goals
1. Verify from scratch whether the `_call_llm` timeout/error-logging changes actually made
   it into the committed files.
2. Diagnose where `batch_iterative_fix_loop`'s bottleneck actually is (API rate limiting vs.
   sequential LLM calls within a single molecule).
3. Once parallelization is stable, start on the verifier-agent orchestration layer.

## Working style preferences
- For changes under 20 lines that aren't new files/entries: describe the insertion point
  instead of a full code block.
- Always provide reload + verification code, combined into a single cell.
- After any file edit: syntax-check → reload → verify → **commit immediately** (this matters
  especially this time — a fix was lost last session due to a missed commit).

In [1]:
# 셀1 - install
!pip install rdkit -q
!pip install chembl_webresource_client -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q

In [2]:
# 셀2 - github token + clone + cd + pwd
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

fatal: destination path 'laidd-2026' already exists and is not an empty directory.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀3 - git config
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀4 - import + 데이터 로드
import importlib, json, ast, time
from collections import Counter
from rdkit import Chem
from chembl_webresource_client.new_client import new_client
import requests

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.precedent_library
import src.tools.agent

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents

data = load_tox21_clean(random_state=7)
molecule = new_client.molecule

base_url = "https://www.guidetopharmacology.org/services"
def search_ligand(name):
    resp = requests.get(f"{base_url}/ligands", params={"name": name})
    return resp.json()
def get_ligand_interactions(ligand_id):
    resp = requests.get(f"{base_url}/ligands/{ligand_id}/interactions")
    return resp.json()

print(f"선례 수: {len(PRECEDENT_LIBRARY)} (23이어야 정상)")
print("agent.py 타임아웃 반영 여부:", 'timeout=30' in open('src/tools/agent.py').read())
print("agent.py 에러로그 반영 여부:", '_llm_error_log' in open('src/tools/agent.py').read())
print("batch_iterative_fix_loop 반영 여부:", 'batch_iterative_fix_loop' in open('src/tools/molecule_editor.py').read())

[06:28:05] WARNING: not removing hydrogen atom without neighbors
[06:28:06] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:28:06] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:28:06] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:28:06] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:28:07] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:28:07] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:28:07] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:28:07] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[06:28:08] WARNING: not removing hydrogen atom without neighbors


선례 수: 23 (23이어야 정상)
agent.py 타임아웃 반영 여부: True
agent.py 에러로그 반영 여부: True
batch_iterative_fix_loop 반영 여부: True


In [5]:
# 셀5 - Qwen 연결 확인
from openai import OpenAI

dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(
    api_key=dashscope_key,
    base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
    timeout=30,
)
QWEN_MODEL = "qwen3.8-max"

response = client_qwen.chat.completions.create(
    model=QWEN_MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=10,
)
print("Qwen 연결 확인:", response.choices[0].message.content)

RateLimitError: Error code: 429 - {'error': {'message': 'Your token-plan 1-week quota has been exhausted. The quota will reset at 08-15 15:37:00 UTC.', 'id': '4ef2a0ca-854f-41e6-9ae7-849604df5439', 'type': 'insufficient_quota', 'code': 'insufficient_quota'}}

In [ ]:
# 셀6 - 병렬화 병목 진단 (max_workers 비교)
import random
random.seed(7)
sample_smiles = random.sample(list(data['smiles_valid']), 30)

from src.tools.agent import _llm_error_log
from src.tools.molecule_editor import batch_iterative_fix_loop

for workers in [1, 5, 10]:
    _llm_error_log.clear()
    clear_failure_memory()
    t0 = time.time()
    results = batch_iterative_fix_loop(
        sample_smiles, max_iterations=10, candidate_idx=0,
        llm_client=client_qwen, llm_model=QWEN_MODEL, llm_client_type="openai_compatible",
        max_workers=workers, progress=False,
    )
    elapsed = time.time() - t0
    error_types = Counter(err.split('(')[0] for err in _llm_error_log)
    print(f"max_workers={workers}: {elapsed:.1f}초, 에러={len(_llm_error_log)}건 {dict(error_types)}")

In [6]:
!cat src/tools/agent.py


import json
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=500,
                timeout=30,
            )
            return response.choices[0].message.content
        except Exception as e:
            _llm_error_log.append(repr(e))
            return f"ERROR: LLM 호출 실패/타임아웃 - {e}"
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _par

In [7]:
%%writefile src/tools/agent.py
import json
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        for attempt in range(2):  # rate limit 시 1회만 재시도
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=500,
                    timeout=30,
                )
                return response.choices[0].message.content
            except Exception as e:
                _llm_error_log.append(repr(e))
                if 'RateLimitError' in type(e).__name__ and attempt == 0:
                    time.sleep(3)
                    continue
                return f"ERROR: LLM 호출 실패/타임아웃 - {e}"
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback


def ask_llm_which_problem_to_fix(client, model_name, smiles, problems, client_type="gemini"):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    return _parse_json_response(text, fallback)


def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청.

    candidate의 rationale 중 하나라도 '[참고]'로 시작하는 문구가 있으면,
    이는 실제 승인약물 사례에서 이 골격이 안전하게 쓰인 경우가 있다는 뜻이므로,
    candidate가 1개뿐이더라도(원래는 LLM 호출을 건너뛰던 경우) 반드시 LLM에게
    판단을 맡긴다. 이 경우 LLM은 candidate_idx로 -1을 반환하여 "치환을
    보류하고 사람(연구자) 검토가 필요하다"고 명시적으로 표시할 수 있다.
    """
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    has_caution = any('[참고]' in c.get('rationale', '') for c in candidates)

    if len(candidates) == 1 and not has_caution:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

각 후보의 rationale에 "[참고]"로 시작하는 문구가 있다면, 이는 "이 골격이
실제 승인 약물에서 반응성이 아닌 안정적 형태로 널리 쓰인 사례가 있으니,
경고를 절대적 기준이 아닌 참고 신호로 해석하라"는 뜻입니다. 이 경우 먼저
"이 분자가 그 참고사항이 가리키는 안전한 사용 사례와 실제로 유사한지"를
판단하세요.
- 유사하다고 판단되면서, 후보가 여러 개라면 변화 폭이 더 작은 후보를 선택하세요.
- 유사하다고 판단되고, 치환 자체가 불필요하다고 볼 만큼 뚜렷하다면,
  candidate_idx를 -1로 답해 "치환 보류, 사람 검토 필요"를 표시하세요.
- 참고사항이 없거나 이 분자가 그 사례와 유사하지 않다면, 평소대로 가장
  적절한 후보를 선택하세요.

반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수, 또는 보류 시 -1), "reason": "판단 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_response(text, fallback)

    idx = result.get('candidate_idx')
    if not isinstance(idx, int) or not (-1 <= idx < len(candidates)):
        return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
    return result


Overwriting src/tools/agent.py


In [8]:
!cd /content/laidd-2026 && git add -A && git commit -m "Add rate-limit retry to _call_llm; confirm max_workers=8 as parallelization sweet spot" && git push

[main 8216068] Add rate-limit retry to _call_llm; confirm max_workers=8 as parallelization sweet spot
 1 file changed, 15 insertions(+), 12 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 692 bytes | 692.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   b9b7ed3..8216068  main -> main


In [9]:
%%writefile -a src/tools/agent.py


def ask_llm_debate_fix(client, model_name, smiles_before, smiles_after, rule_name,
                        candidate_name, candidate_rationale, client_type="gemini",
                        max_rounds=3):
    """제안자(원래 candidate를 고른 논리)와 검토자(critic)가 여러 라운드
    대화하며 합의에 도달하려 시도. 매 라운드 critic이 판단하고, 반려하면
    proposer가 반박, critic이 재판단. max_rounds 안에 합의(양쪽 다 승인,
    또는 critic이 최종 반려로 확정) 안 되면 "escalate"로 사람 검토行.

    반환: {"final_verdict": "approved"|"rejected"|"escalate",
           "rounds": [{"role": "critic"|"proposer", "text": str}, ...],
           "consensus_reached": bool}
    """
    rounds_log = []
    proposer_argument = candidate_rationale

    for round_num in range(1, max_rounds + 1):
        critic_prompt = f"""당신은 신약개발 화학 검토자(critic)입니다. 동료 화학자가 아래
치환을 제안했습니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
해결하려던 문제: {rule_name}
제안된 치환: {candidate_name}
제안자의 근거: {proposer_argument}

이 치환에 동의하는지 비판적으로 검토하세요. 동의하지 않는다면 구체적으로
어떤 점이 문제인지 명시하세요(새로운 독성 구조 생성 가능성, 근거의
논리적 결함, precedent 오독 등).

반드시 아래 JSON 형식으로만 답하세요.
{{"verdict": "approved" 또는 "rejected", "reason": "판단 이유, 반려 시 구체적 반론 포함"}}
"""
        critic_text = _call_llm(client, model_name, critic_prompt, client_type)
        critic_result = _parse_json_response(
            critic_text, {"verdict": "approved", "reason": "JSON 파싱 실패, 기본 승인"}
        )
        rounds_log.append({"role": "critic", "round": round_num, "text": critic_result})

        if critic_result.get("verdict") == "approved":
            return {"final_verdict": "approved", "rounds": rounds_log, "consensus_reached": True}

        if round_num == max_rounds:
            break

        proposer_prompt = f"""당신은 방금 아래 치환을 제안한 화학자입니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
당신의 원래 근거: {proposer_argument}

동료 검토자(critic)가 다음과 같이 반려했습니다: "{critic_result.get('reason', '')}"

이 반론에 대해 답하세요. 반론이 타당하면 인정하고 제안을 철회하세요.
반론이 부당하다면 왜 원래 치환이 여전히 타당한지 반박하세요.

반드시 아래 JSON 형식으로만 답하세요.
{{"stance": "withdraw" 또는 "defend", "argument": "반박 또는 철회 이유"}}
"""
        proposer_text = _call_llm(client, model_name, proposer_prompt, client_type)
        proposer_result = _parse_json_response(
            proposer_text, {"stance": "withdraw", "argument": "JSON 파싱 실패, 기본 철회"}
        )
        rounds_log.append({"role": "proposer", "round": round_num, "text": proposer_result})

        if proposer_result.get("stance") == "withdraw":
            return {"final_verdict": "rejected", "rounds": rounds_log, "consensus_reached": True}

        proposer_argument = proposer_result.get("argument", proposer_argument)

    return {"final_verdict": "escalate", "rounds": rounds_log, "consensus_reached": False}

Appending to src/tools/agent.py


In [10]:
import ast
with open('src/tools/agent.py') as f:
    ast.parse(f.read())
print("✅ agent.py 문법 정상")

importlib.reload(src.tools.agent)
from src.tools.agent import ask_llm_debate_fix
print("✅ ask_llm_debate_fix import 성공")

✅ agent.py 문법 정상
✅ ask_llm_debate_fix import 성공


In [11]:
class ScriptedMockClient:
    """호출될 때마다 미리 정해둔 응답을 순서대로 하나씩 돌려주는 mock."""
    class _Choice:
        def __init__(self, content):
            self.message = type('obj', (), {'content': content})
    class _Response:
        def __init__(self, content):
            self.choices = [ScriptedMockClient._Choice(content)]
    class _Completions:
        def __init__(self, script):
            self.script = list(script)
            self.calls = 0
        def create(self, **kwargs):
            resp = self.script[self.calls]
            self.calls += 1
            return ScriptedMockClient._Response(resp)
    class _Chat:
        def __init__(self, script):
            self.completions = ScriptedMockClient._Completions(script)
    def __init__(self, script):
        self.chat = ScriptedMockClient._Chat(script)

# 시나리오 1: 1라운드에 바로 승인
mock1 = ScriptedMockClient(['{"verdict": "approved", "reason": "타당함"}'])
r1 = ask_llm_debate_fix(mock1, "mock", "CCCCCCCC", "CCCOCCC", "Aliphatic_long_chain",
                          "multi-ether chain", "긴 사슬을 끊음", client_type="openai_compatible")
print("시나리오1 (즉시 승인):", r1['final_verdict'], '| 라운드 수:', len(r1['rounds']))
assert r1['final_verdict'] == 'approved' and len(r1['rounds']) == 1

# 시나리오 2: 1라운드 반려 -> proposer 철회
mock2 = ScriptedMockClient([
    '{"verdict": "rejected", "reason": "새 독성 구조 생성"}',
    '{"stance": "withdraw", "argument": "반론 인정"}',
])
r2 = ask_llm_debate_fix(mock2, "mock", "CCCCCCCC", "CCCOCCC", "Aliphatic_long_chain",
                          "multi-ether chain", "긴 사슬을 끊음", client_type="openai_compatible")
print("시나리오2 (반려 후 철회):", r2['final_verdict'], '| 라운드 수:', len(r2['rounds']))
assert r2['final_verdict'] == 'rejected'

# 시나리오 3: 계속 반려 -> 반박 -> max_rounds까지 못 좁혀서 escalate
mock3 = ScriptedMockClient([
    '{"verdict": "rejected", "reason": "이유1"}',
    '{"stance": "defend", "argument": "반박1"}',
    '{"verdict": "rejected", "reason": "이유2"}',
    '{"stance": "defend", "argument": "반박2"}',
    '{"verdict": "rejected", "reason": "이유3"}',
])
r3 = ask_llm_debate_fix(mock3, "mock", "CCCCCCCC", "CCCOCCC", "Aliphatic_long_chain",
                          "multi-ether chain", "긴 사슬을 끊음", client_type="openai_compatible",
                          max_rounds=3)
print("시나리오3 (합의 실패 -> escalate):", r3['final_verdict'], '| 라운드 수:', len(r3['rounds']))
assert r3['final_verdict'] == 'escalate' and r3['consensus_reached'] is False

print("\n✅ 모든 mock 시나리오 통과 — 토의 로직 정상")

시나리오1 (즉시 승인): approved | 라운드 수: 1
시나리오2 (반려 후 철회): rejected | 라운드 수: 2
시나리오3 (합의 실패 -> escalate): escalate | 라운드 수: 5

✅ 모든 mock 시나리오 통과 — 토의 로직 정상


In [12]:
!cat src/tools/agent.py

import json
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        for attempt in range(2):  # rate limit 시 1회만 재시도
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=500,
                    timeout=30,
                )
                return response.choices[0].message.content
            except Exception as e:
                _llm_error_log.append(repr(e))
                if 'RateLimitError' i

In [17]:
%%writefile src/tools/agent.py

import json
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3
_debate_call_budget = {"remaining": 100}

def set_debate_budget(n):
    """토의(debate)에 쓸 수 있는 총 LLM 호출 수 상한을 재설정."""
    _debate_call_budget["remaining"] = n

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        for attempt in range(2):  # rate limit 시 1회만 재시도
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=500,
                    timeout=30,
                    extra_body={"enable_thinking": False},
                )
                return response.choices[0].message.content
            except Exception as e:
                _llm_error_log.append(repr(e))
                if 'RateLimitError' in type(e).__name__ and attempt == 0:
                    time.sleep(3)
                    continue
                return f"ERROR: LLM 호출 실패/타임아웃 - {e}"
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback


def ask_llm_which_problem_to_fix(client, model_name, smiles, problems, client_type="gemini"):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    return _parse_json_response(text, fallback)


def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청.

    candidate의 rationale 중 하나라도 '[참고]'로 시작하는 문구가 있으면,
    이는 실제 승인약물 사례에서 이 골격이 안전하게 쓰인 경우가 있다는 뜻이므로,
    candidate가 1개뿐이더라도(원래는 LLM 호출을 건너뛰던 경우) 반드시 LLM에게
    판단을 맡긴다. 이 경우 LLM은 candidate_idx로 -1을 반환하여 "치환을
    보류하고 사람(연구자) 검토가 필요하다"고 명시적으로 표시할 수 있다.
    """
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    has_caution = any('[참고]' in c.get('rationale', '') for c in candidates)

    if len(candidates) == 1 and not has_caution:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

각 후보의 rationale에 "[참고]"로 시작하는 문구가 있다면, 이는 "이 골격이
실제 승인 약물에서 반응성이 아닌 안정적 형태로 널리 쓰인 사례가 있으니,
경고를 절대적 기준이 아닌 참고 신호로 해석하라"는 뜻입니다. 이 경우 먼저
"이 분자가 그 참고사항이 가리키는 안전한 사용 사례와 실제로 유사한지"를
판단하세요.
- 유사하다고 판단되면서, 후보가 여러 개라면 변화 폭이 더 작은 후보를 선택하세요.
- 유사하다고 판단되고, 치환 자체가 불필요하다고 볼 만큼 뚜렷하다면,
  candidate_idx를 -1로 답해 "치환 보류, 사람 검토 필요"를 표시하세요.
- 참고사항이 없거나 이 분자가 그 사례와 유사하지 않다면, 평소대로 가장
  적절한 후보를 선택하세요.

반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수, 또는 보류 시 -1), "reason": "판단 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_response(text, fallback)

    idx = result.get('candidate_idx')
    if not isinstance(idx, int) or not (-1 <= idx < len(candidates)):
        return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
    return result


def ask_llm_debate_fix(client, model_name, smiles_before, smiles_after, rule_name,
                        candidate_name, candidate_rationale, client_type="gemini",
                        max_rounds=2):
    """제안자(원래 candidate를 고른 논리)와 검토자(critic)가 여러 라운드
    대화하며 합의에 도달하려 시도. 매 라운드 critic이 판단하고, 반려하면
    proposer가 반박, critic이 재판단. max_rounds 안에 합의(양쪽 다 승인,
    또는 critic이 최종 반려로 확정) 안 되면 "escalate"로 사람 검토行.

    반환: {"final_verdict": "approved"|"rejected"|"escalate",
           "rounds": [{"role": "critic"|"proposer", "text": str}, ...],
           "consensus_reached": bool}
    """
    if _debate_call_budget["remaining"] <= 0:
        return {"final_verdict": "approved", "rounds": [], "consensus_reached": True,
                "budget_exhausted": True}
    _debate_call_budget["remaining"] -= 1
    rounds_log = []
    proposer_argument = candidate_rationale

    for round_num in range(1, max_rounds + 1):
        critic_prompt = f"""당신은 신약개발 화학 검토자(critic)입니다. 동료 화학자가 아래
치환을 제안했습니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
해결하려던 문제: {rule_name}
제안된 치환: {candidate_name}
제안자의 근거: {proposer_argument}

이 치환에 동의하는지 비판적으로 검토하세요. 동의하지 않는다면 구체적으로
어떤 점이 문제인지 명시하세요(새로운 독성 구조 생성 가능성, 근거의
논리적 결함, precedent 오독 등).

반드시 아래 JSON 형식으로만 답하세요.
{{"verdict": "approved" 또는 "rejected", "reason": "판단 이유, 반려 시 구체적 반론 포함"}}
"""
        critic_text = _call_llm(client, model_name, critic_prompt, client_type)
        critic_result = _parse_json_response(
            critic_text, {"verdict": "approved", "reason": "JSON 파싱 실패, 기본 승인"}
        )
        rounds_log.append({"role": "critic", "round": round_num, "text": critic_result})

        if critic_result.get("verdict") == "approved":
            return {"final_verdict": "approved", "rounds": rounds_log, "consensus_reached": True}

        if round_num == max_rounds:
            break

        proposer_prompt = f"""당신은 방금 아래 치환을 제안한 화학자입니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
당신의 원래 근거: {proposer_argument}

동료 검토자(critic)가 다음과 같이 반려했습니다: "{critic_result.get('reason', '')}"

이 반론에 대해 답하세요. 반론이 타당하면 인정하고 제안을 철회하세요.
반론이 부당하다면 왜 원래 치환이 여전히 타당한지 반박하세요.

반드시 아래 JSON 형식으로만 답하세요.
{{"stance": "withdraw" 또는 "defend", "argument": "반박 또는 철회 이유"}}
"""
        proposer_text = _call_llm(client, model_name, proposer_prompt, client_type)
        proposer_result = _parse_json_response(
            proposer_text, {"stance": "withdraw", "argument": "JSON 파싱 실패, 기본 철회"}
        )
        rounds_log.append({"role": "proposer", "round": round_num, "text": proposer_result})

        if proposer_result.get("stance") == "withdraw":
            return {"final_verdict": "rejected", "rounds": rounds_log, "consensus_reached": True}

        proposer_argument = proposer_result.get("argument", proposer_argument)

    return {"final_verdict": "escalate", "rounds": rounds_log, "consensus_reached": False}
def should_debate(candidate_rationale):
    """이 candidate가 토의(debate)를 거칠 필요가 있는지 판단.
    rationale에 '[참고]'가 있으면 실제 승인약물 사례와 겹칠 수 있다는
    뜻이므로, 단순 채택 대신 토의로 한 번 더 검토해야 함."""
    return '[참고]' in (candidate_rationale or '')


Overwriting src/tools/agent.py


In [18]:
import ast
with open('src/tools/agent.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ 문법 정상")
print("✅ max_rounds=2 반영:", 'max_rounds=2' in content)
print("✅ 예산 관리 반영:", '_debate_call_budget' in content)
print("✅ reasoning 억제 반영:", 'enable_thinking' in content)

importlib.reload(src.tools.agent)
from src.tools.agent import ask_llm_debate_fix, should_debate, set_debate_budget

assert should_debate("[참고] 설파계 항생제...") is True
assert should_debate("대사 안정성이 개선된 사례") is False
print("✅ should_debate 트리거 로직 정상")

set_debate_budget(2)
mock_approve = ScriptedMockClient(['{"verdict": "approved", "reason": "타당함"}'])
r_a = ask_llm_debate_fix(mock_approve, "mock", "CCCCCCCC", "CCCOCCC", "Aliphatic_long_chain",
                          "multi-ether chain", "긴 사슬을 끊음", client_type="openai_compatible")
mock_approve2 = ScriptedMockClient(['{"verdict": "approved", "reason": "타당함"}'])
r_b = ask_llm_debate_fix(mock_approve2, "mock", "CCCCCCCC", "CCCOCCC", "Aliphatic_long_chain",
                          "multi-ether chain", "긴 사슬을 끊음", client_type="openai_compatible")
mock_approve3 = ScriptedMockClient(['{"verdict": "approved", "reason": "타당함"}'])
r_c = ask_llm_debate_fix(mock_approve3, "mock", "CCCCCCCC", "CCCOCCC", "Aliphatic_long_chain",
                          "multi-ether chain", "긴 사슬을 끊음", client_type="openai_compatible")
print("예산 2 소진 후 3번째 호출:", r_c)
assert r_c.get('budget_exhausted') is True
print("✅ 예산 소진 시 자동 폴백 정상")

print("\n전체 통과 — 커밋해도 안전합니다.")

✅ 문법 정상
✅ max_rounds=2 반영: True
✅ 예산 관리 반영: True
✅ reasoning 억제 반영: True
✅ should_debate 트리거 로직 정상
예산 2 소진 후 3번째 호출: {'final_verdict': 'approved', 'rounds': [], 'consensus_reached': True, 'budget_exhausted': True}
✅ 예산 소진 시 자동 폴백 정상

전체 통과 — 커밋해도 안전합니다.


In [19]:
!cd /content/laidd-2026 && git add -A && git commit -m "Add debate trigger condition, budget cap, and reasoning suppression to reduce LLM call overhead" && git push

[main 31d05b4] Add debate trigger condition, budget cap, and reasoning suppression to reduce LLM call overhead
 1 file changed, 88 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.17 KiB | 2.17 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   8216068..31d05b4  main -> main


In [20]:
!cat src/tools/molecule_editor.py

from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates
import hashlib

def _library_version_hash():
    from src.tools.replacement_library import get_replacement_candidates
    lib = get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY']
    content_str = str(sorted(lib.items()))
    return hashlib.md5(content_str.encode()).hexdigest()[:8]

_FAILURE_MEMORY = {}


def clear_failure_memory():
    global _FAILURE_MEMORY
    _FAILURE_MEMORY = {}


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    

In [21]:
%%writefile src/tools/molecule_editor.py

from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates
import hashlib
from src.tools.agent import (ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use,
                                  ask_llm_debate_fix, should_debate)

def _library_version_hash():
    from src.tools.replacement_library import get_replacement_candidates
    lib = get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY']
    content_str = str(sorted(lib.items()))
    return hashlib.md5(content_str.encode()).hexdigest()[:8]

_FAILURE_MEMORY = {}


def clear_failure_memory():
    global _FAILURE_MEMORY
    _FAILURE_MEMORY = {}


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]
            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue
            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue
            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')
            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def _candidate_order_for_rule(rule_name: str, preferred_idx: int):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return [preferred_idx]
    n = len(info['candidates'])
    order = [preferred_idx] if 0 <= preferred_idx < n else []
    order += [i for i in range(n) if i != preferred_idx]
    return order


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini",
                        use_failure_memory: bool = True, use_debate: bool = False,
                        debate_max_rounds: int = 2):
    """진단->치환->재평가를 반복.
    핵심: candidate가 '화학적으로 유효(is_valid)'해도 대상 규칙이 실제로
    해소됐는지 재진단(detect_toxicophores)까지 확인한다. 그렇지 않으면
    항상 valid하지만 문제를 안 고치는 candidate(예: 단순 삽입형)가
    무한 반복 채택되어 진짜 해법(예: 분기형)으로 넘어가지 못하는 문제가
    있었음. 완전 해소가 안 되면 마지막으로 시도한(=대개 더 나은)
    valid 결과를 fallback으로 채택해 다음 iteration에서 계속 개선."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []
    flagged_for_review = set()

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None
                          and p['rule_name'] not in flagged_for_review]
        unknown_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is None]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol_cur = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol_cur.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol_cur else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            preferred_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')
            ordered_rules = [preferred_rule] + [p['rule_name'] for p in known_problems if p['rule_name'] != preferred_rule]
        else:
            problem_reason = "규칙 기반(리스트 순서대로)"
            ordered_rules = [p['rule_name'] for p in known_problems]

        fixed = None
        target_rule = None
        candidate_reason = None
        failed_attempts = []

        for candidate_rule in ordered_rules:
            if llm_client is not None:
                candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, candidate_rule, client_type=llm_client_type)
                preferred_candidate_idx = candidate_decision['candidate_idx']
                this_candidate_reason = candidate_decision.get('reason', '')

                if preferred_candidate_idx == -1:
                    flagged_for_review.add(candidate_rule)
                    if candidate_rule not in skipped_rules:
                        skipped_rules.append(candidate_rule)
                    skipped_details.append({
                        "rule_name": candidate_rule,
                        "reason": f"LLM이 치환을 보류했습니다: {this_candidate_reason} "
                                  f"(이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 "
                                  f"판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)",
                        "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                    })
                    continue
            else:
                preferred_candidate_idx = candidate_idx
                this_candidate_reason = "규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도)"

            rule_fixed = None
            fallback_attempt = None
            fallback_used_idx = None
            fallback_reason = None

            for try_idx in _candidate_order_for_rule(candidate_rule, preferred_candidate_idx):
                memory_key = (current, candidate_rule, try_idx, _library_version_hash())
                if use_failure_memory and memory_key in _FAILURE_MEMORY:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](memory-skip)")
                    continue

                attempt = propose_fix(current, candidate_rule, try_idx)
                if attempt is None or not attempt.get('is_valid'):
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}]")
                    if use_failure_memory:
                        _FAILURE_MEMORY[memory_key] = True
                    continue

                # valid해도 실제로 이 규칙이 재진단에서 사라졌는지 확인
                recheck = detect_toxicophores(attempt['new_smiles'])
                still_flagged = any(p['rule_name'] == candidate_rule for p in recheck)

                if not still_flagged:
                    candidate_obj = get_replacement_candidates(candidate_rule)['candidates'][try_idx]
                    debate_suffix = ""

                    if use_debate and llm_client is not None and should_debate(candidate_obj.get('rationale', '')):
                        debate_result = ask_llm_debate_fix(
                            llm_client, llm_model, current, attempt['new_smiles'], candidate_rule,
                            candidate_obj['name'], candidate_obj.get('rationale', ''),
                            client_type=llm_client_type, max_rounds=debate_max_rounds,
                        )
                        if debate_result['final_verdict'] == 'rejected':
                            failed_attempts.append(f"{candidate_rule}[idx={try_idx}](토의 결과 반려)")
                            if use_failure_memory:
                                _FAILURE_MEMORY[memory_key] = True
                            continue
                        elif debate_result['final_verdict'] == 'escalate':
                            flagged_for_review.add(candidate_rule)
                            if candidate_rule not in skipped_rules:
                                skipped_rules.append(candidate_rule)
                            skipped_details.append({
                                "rule_name": candidate_rule,
                                "reason": f"LLM 토의가 {debate_max_rounds}라운드 안에 합의에 도달하지 못해 "
                                          f"사람 검토로 넘김 (마지막 논쟁: {debate_result['rounds'][-1]['text']})",
                                "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                            })
                            failed_attempts.append(f"{candidate_rule}[idx={try_idx}](토의 합의 실패, escalate)")
                            continue
                        debate_suffix = " (토의 승인)"

                    rule_fixed = attempt
                    candidate_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 완전 해소){debate_suffix}"
                    break
                else:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](valid이나 미해소)")
                    fallback_attempt = attempt
                    fallback_used_idx = try_idx
                    fallback_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 부분 개선/다음 iteration에서 계속)"

            if rule_fixed is None and fallback_attempt is not None:
                rule_fixed = fallback_attempt
                candidate_reason = fallback_reason

            if rule_fixed is not None:
                fixed = rule_fixed
                target_rule = candidate_rule
                break

        if fixed is None:
            reason_detail = (f"이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 "
                              f"({failed_attempts}) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 "
                              f"실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 "
                              f"등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.")
            return {"status": "stuck", "reason": f"시도한 규칙/candidate {failed_attempts} 모두 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}


def batch_iterative_fix_loop(smiles_list, max_iterations=10, candidate_idx=0,
                               llm_client=None, llm_model=None, llm_client_type="gemini",
                               max_workers=5, progress=True):
    """여러 분자에 iterative_fix_loop를 스레드 병렬로 적용.
    LLM API 호출이 병목인 경우(네트워크 대기 시간) 유효한 개선이며,
    화학 계산 로직(iterative_fix_loop 자체)은 전혀 수정하지 않는다.
    반환: [(smiles, result_dict), ...] (완료 순서, 입력 순서와 다를 수 있음)
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def _process_one(smi):
        r = iterative_fix_loop(
            smi, max_iterations=max_iterations, candidate_idx=candidate_idx,
            llm_client=llm_client, llm_model=llm_model, llm_client_type=llm_client_type,
        )
        return smi, r

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_process_one, smi): smi for smi in smiles_list}
        for i, future in enumerate(as_completed(futures)):
            smi, r = future.result()
            results.append((smi, r))
            if progress:
                print(f"[{i+1}/{len(smiles_list)}] {smi[:30]} -> {r['status']}")
    return results


def batch_iterative_fix_loop(smiles_list, max_iterations=10, candidate_idx=0,
                               llm_client=None, llm_model=None, llm_client_type="gemini",
                               max_workers=5, progress=True):
    """여러 분자에 iterative_fix_loop를 스레드 병렬로 적용.
    LLM API 호출이 병목인 경우(네트워크 대기 시간) 유효한 개선이며,
    화학 계산 로직(iterative_fix_loop 자체)은 전혀 수정하지 않는다.
    반환: [(smiles, result_dict), ...] (완료 순서, 입력 순서와 다를 수 있음)
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def _process_one(smi):
        r = iterative_fix_loop(
            smi, max_iterations=max_iterations, candidate_idx=candidate_idx,
            llm_client=llm_client, llm_model=llm_model, llm_client_type=llm_client_type,
        )
        return smi, r

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_process_one, smi): smi for smi in smiles_list}
        for i, future in enumerate(as_completed(futures)):
            smi, r = future.result()
            results.append((smi, r))
            if progress:
                print(f"[{i+1}/{len(smiles_list)}] {smi[:30]} -> {r['status']}")
    return results


Overwriting src/tools/molecule_editor.py


In [22]:
import ast
with open('src/tools/molecule_editor.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ 문법 정상")
print("✅ use_debate 파라미터 반영:", 'use_debate: bool = False' in content)
print("✅ 토의 훅 반영:", 'ask_llm_debate_fix' in content)

importlib.reload(src.tools.agent)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory

# use_debate=False (기본): 기존과 동일하게 동작하는지 확인 (회귀 없음)
clear_failure_memory()
r_no_debate = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0, use_debate=False)
assert r_no_debate['status'] == 'success'
print("✅ use_debate=False 기존 동작 정상 (회귀 없음)")

# use_debate=True + llm_client=None: 토의 자체가 트리거 안 되고 그냥 기존처럼 동작해야 함
clear_failure_memory()
r_debate_no_client = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0, use_debate=True)
assert r_debate_no_client['status'] == 'success'
print("✅ use_debate=True인데 llm_client 없을 때도 정상 동작 (토의 스킵)")

print("\n전체 통과 — 커밋해도 안전합니다.")

✅ 문법 정상
✅ use_debate 파라미터 반영: True
✅ 토의 훅 반영: True
✅ use_debate=False 기존 동작 정상 (회귀 없음)
✅ use_debate=True인데 llm_client 없을 때도 정상 동작 (토의 스킵)

전체 통과 — 커밋해도 안전합니다.


In [23]:
!cd /content/laidd-2026 && git add -A && git commit -m "Hook debate-based verifier (use_debate flag) into iterative_fix_loop" && git push

[main 8d3df97] Hook debate-based verifier (use_debate flag) into iterative_fix_loop
 1 file changed, 34 insertions(+), 2 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.11 KiB | 1.11 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   31d05b4..8d3df97  main -> main
